# Customer Churn Predictor
**Goal:** Predict which B2B SaaS customers will churn in 90 days and explain why.

**Dataset:** 1,500 synthetic accounts with deliberately messy data (string booleans, mixed NPS values, 15-20% missing fields)

**Workflow:** EDA → Cleaning → Feature Engineering → Model → SHAP Explainability

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import roc_auc_score, classification_report, roc_curve, confusion_matrix
from xgboost import XGBClassifier
import shap

sns.set_theme(style='whitegrid', font_scale=1.1)
plt.rcParams.update({'figure.dpi':110,'axes.titleweight':'bold'})
print('All imports OK ✓')

## 1 · Load & Audit Data

In [ ]:
df = pd.read_csv('churn_data.csv')
print(f'Shape: {df.shape}')
print(f'Churn rate: {df["churned_90d"].mean():.1%}')
print('\nData types and null counts:')
for col in df.columns:
    nulls = df[col].isna().sum()
    pct = nulls/len(df)*100
    print(f'  {col:25s} {str(df[col].dtype):10s} nulls:{nulls:4d} ({pct:.1f}%)')

In [ ]:
# Spot the messy columns
print('onboarding_complete unique values:', df['onboarding_complete'].unique())
print('nps_score unique values (sample):', df['nps_score'].unique()[:15])
print('company_size unique:', df['company_size'].unique())
print('cs_health_score nulls:', df['cs_health_score'].isna().sum())

## 2 · EDA

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14,8))
fig.suptitle('Key Feature Distributions by Churn Outcome', fontweight='bold')

numeric_features = ['login_count_30d','feature_adoption_pct','last_login_days_ago','support_tickets_90d','tenure_months','billing_failures_6m']

for ax, feat in zip(axes.flat, numeric_features):
    for label, color, name in [(0,'#4caf7d','Retained'),(1,'#e05c5c','Churned')]:
        subset = df[df['churned_90d']==label][feat]
        sns.kdeplot(subset, ax=ax, fill=True, color=color, label=name, alpha=0.45)
    ax.set_title(feat)
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig('eda_distributions.png', bbox_inches='tight')
plt.show()

In [ ]:
# Churn rate by plan
plan_churn = df.groupby('plan')['churned_90d'].agg(['mean','count']).reset_index()
plan_churn.columns = ['plan','churn_rate','count']
plan_churn = plan_churn.sort_values('churn_rate',ascending=False)

fig, ax = plt.subplots(figsize=(8,4))
ax.bar(plan_churn['plan'], plan_churn['churn_rate']*100, color=sns.color_palette('Reds_r',4), edgecolor='white')
ax.set_title('Churn Rate by Plan')
ax.set_ylabel('Churn Rate (%)')
for i, row in plan_churn.iterrows():
    ax.text(list(plan_churn['plan']).index(row['plan']), row['churn_rate']*100+0.3, f"{row['churn_rate']*100:.1f}%", ha='center')
plt.tight_layout()
plt.savefig('churn_by_plan.png', bbox_inches='tight')
plt.show()

## 3 · Data Cleaning

Every decision documented with reasoning.

In [ ]:
df_clean = df.copy()

# WHY: onboarding_complete has 6 string variants from different data entry sources
# DECISION: map all truthy variants to 1, falsy to 0
bool_map = {'True':1,'true':1,'1':1,'False':0,'false':0,'0':0}
df_clean['onboarding_complete'] = df_clean['onboarding_complete'].map(bool_map).fillna(0).astype(int)
print('onboarding_complete fixed:', df_clean['onboarding_complete'].value_counts().to_dict())

# WHY: nps_score has string scores, 'N/A', and 'MISSING'
# DECISION: convert to numeric, create missing flag, impute with median
# REASONING: missingness itself is a signal (disengaged customers skip surveys)
df_clean['nps_score'] = pd.to_numeric(df_clean['nps_score'].replace({'N/A':np.nan,'MISSING':np.nan}), errors='coerce')
df_clean['nps_missing'] = df_clean['nps_score'].isna().astype(int)
df_clean['nps_score'] = df_clean['nps_score'].fillna(df_clean['nps_score'].median())
print(f'NPS: {df_clean["nps_missing"].sum()} missing values flagged and imputed')

# WHY: company_size has 'MISSING' string
# DECISION: replace with 'Unknown' category, not drop
# REASONING: losing 5% of rows would reduce training data without clear benefit
df_clean['company_size'] = df_clean['company_size'].replace('MISSING','Unknown')

# WHY: cs_health_score has 15% nulls
# DECISION: flag + median impute (same as NPS)
df_clean['cs_health_missing'] = df_clean['cs_health_score'].isna().astype(int)
df_clean['cs_health_score'] = df_clean['cs_health_score'].fillna(df_clean['cs_health_score'].median())
print(f'CS health: {df_clean["cs_health_missing"].sum()} nulls flagged and imputed')

# Encode categoricals
size_map = {'Small':1,'Medium':2,'Large':3,'Enterprise':4,'Unknown':0}
plan_map = {'Starter':1,'Pro':2,'Business':3,'Enterprise':4}
df_clean['company_size_enc'] = df_clean['company_size'].map(size_map).fillna(0)
df_clean['plan_enc'] = df_clean['plan'].map(plan_map).fillna(1)

print('\nCleaning complete. No remaining nulls in features:', df_clean[FEATURES].isna().sum().sum())


## 4 · Model Training

In [ ]:
FEATURES = [
    'tenure_months','login_count_30d','feature_adoption_pct',
    'support_tickets_90d','nps_score','nps_missing',
    'contract_end_days','last_login_days_ago','billing_failures_6m',
    'integrations_active','onboarding_complete','cs_health_score',
    'cs_health_missing','monthly_revenue','company_size_enc','plan_enc'
]

X = df_clean[FEATURES]
y = df_clean['churned_90d']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'Train: {X_train.shape} | Test: {X_test.shape}')

In [ ]:
# WHY XGBoost?
# 1. Handles mixed feature types well
# 2. Built-in feature importance for explainability
# 3. scale_pos_weight handles class imbalance without oversampling
# 4. Works well with SHAP for stakeholder explanations

model = XGBClassifier(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=(y_train==0).sum()/(y_train==1).sum(),
    random_state=42,
    eval_metric='logloss'
)

# 5-fold CV
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(model, X_train, y_train, cv=cv, scoring='roc_auc')
print(f'CV ROC-AUC: {cv_scores.mean():.4f} +/- {cv_scores.std():.4f}')

model.fit(X_train, y_train)
y_prob = model.predict_proba(X_test)[:,1]
test_auc = roc_auc_score(y_test, y_prob)
print(f'Test ROC-AUC: {test_auc:.4f}')

In [ ]:
# ROC Curve
fpr, tpr, thresholds = roc_curve(y_test, y_prob)

fig, axes = plt.subplots(1, 2, figsize=(13,5))

axes[0].plot(fpr, tpr, color='#e05c5c', lw=2, label=f'ROC (AUC={test_auc:.3f})')
axes[0].plot([0,1],[0,1],'k--',alpha=0.4)
axes[0].set_title('ROC Curve')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].legend()

y_pred = (y_prob >= 0.4).astype(int)
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Reds', ax=axes[1],
            xticklabels=['Retained','Churned'], yticklabels=['Retained','Churned'])
axes[1].set_title('Confusion Matrix (threshold=0.4)')
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('Actual')

plt.tight_layout()
plt.savefig('model_performance.png', bbox_inches='tight')
plt.show()
print(classification_report(y_test, y_pred, target_names=['Retained','Churned']))

## 5 · SHAP Explainability

In [ ]:
explainer   = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test)

plt.figure(figsize=(10,7))
shap.summary_plot(shap_values, X_test, feature_names=FEATURES, show=False)
plt.title('SHAP Feature Importance — What Drives Churn?', fontweight='bold')
plt.tight_layout()
plt.savefig('shap_summary.png', bbox_inches='tight')
plt.show()

In [ ]:
# Waterfall for a single high-risk account
high_risk_idx = y_prob.argmax()
print(f'Highest risk account: churn probability = {y_prob[high_risk_idx]:.1%}')
print(X_test.iloc[high_risk_idx])

shap_exp = shap.Explanation(
    values=shap_values[high_risk_idx],
    base_values=explainer.expected_value,
    data=X_test.iloc[high_risk_idx].values,
    feature_names=FEATURES
)
plt.figure(figsize=(10,5))
shap.waterfall_plot(shap_exp, show=False)
plt.title('Why is this account high risk?', fontweight='bold')
plt.tight_layout()
plt.savefig('shap_waterfall.png', bbox_inches='tight')
plt.show()

## 6 · Save Model

In [ ]:
with open('model_artifacts.pkl','wb') as f:
    pickle.dump({'model':model,'features':FEATURES,'threshold':0.4,'auc':test_auc}, f)
print('model_artifacts.pkl saved ✓')
print(f'Final Test ROC-AUC: {test_auc:.4f}')